# Table 7 — Cross-validation Against PCD Reference Stations

**Script file:** `Paper1_Table7_pcd_crossvalidation.ipynb`

สร้าง **Table 7** ของต้นฉบับ (Correlation between field-measured PM2.5 and same-day PM2.5 at the nearest PCD reference station) — เปรียบเทียบ PM2.5 ที่วัดเองภาคสนามกับค่าเฉลี่ยรายวันของสถานี PCD ที่ใกล้ที่สุด (24T หรือ 25T)

**สิ่งที่ต้องเตรียมก่อนรัน:**
1. ไฟล์ `บันทึกการตรวจวัดปริมาณฝุ่น.xlsx` (ข้อมูลภาคสนามต้นฉบับ 32 จุด — ไฟล์เดียวกับที่ใช้ใน Table1/Table2/Figure1)
2. ไฟล์ export จาก Air4Thai (เช่น `2023.xlsx`) — ไฟล์เดียว รวมทุกสถานีทั้งเครือข่าย ชีตชื่อ `PM2.5`, คอลัมน์ `Date` + รหัสสถานี (`24T`, `25T`, ...) เป็นค่าเฉลี่ยรายวัน ดาวน์โหลดได้จาก https://air4thai.pcd.go.th/webV3/#/History

## 1. ติดตั้งไลบรารีและ import

In [ ]:
!pip install -q requests pandas numpy scipy matplotlib openpyxl

import re, time
import numpy as np
import pandas as pd
import requests
import openpyxl
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import files

plt.rcParams['figure.dpi'] = 120
print('พร้อมใช้งาน ✅')

## 2. อัปโหลดข้อมูลภาคสนามต้นฉบับ และแยกข้อมูล

In [ ]:
from google.colab import files
print('อัปโหลดไฟล์ "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx":')
uploaded = files.upload()
RAW_XLSX = list(uploaded.keys())[0]
print('อัปโหลดไฟล์แล้ว:', RAW_XLSX)

In [ ]:
wb = openpyxl.load_workbook(RAW_XLSX, data_only=True)
ws = wb['Sheet1']

# convention เดิม (column index) จาก Table1/Table2/Figure1 notebooks
COL_NAME, COL_URL, COL_ADDRESS, COL_PM25, COL_DATE = 0, 1, 2, 6, 13

records = {}
order = []
for row in ws.iter_rows(min_row=3, max_row=1001, values_only=True):
    name = row[COL_NAME]
    if not name:
        continue
    if name not in records:
        records[name] = {
            'name': name, 'url': row[COL_URL], 'address': row[COL_ADDRESS],
            'pm25': row[COL_PM25], 'date': str(row[COL_DATE]),
        }
        order.append(name)

sites_raw = [records[n] for n in order]
print(f'จำนวนจุดตรวจวัดทั้งหมด: {len(sites_raw)}')
assert len(sites_raw) == 32, 'คาดว่าจะมี 32 จุด - ตรวจสอบไฟล์ต้นฉบับถ้าไม่ตรง'

## 3. Resolve พิกัด GPS ของจุดภาคสนาม

In [ ]:
PATTERNS = [
    re.compile(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)"),
    re.compile(r"@(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)"),
    re.compile(r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"),
]

def extract_latlon(url_or_text):
    for pat in PATTERNS:
        m = pat.search(url_or_text)
        if m:
            return float(m.group(1)), float(m.group(2))
    return None, None

def resolve_link(short_url, timeout=15):
    try:
        r = requests.get(short_url, allow_redirects=True, timeout=timeout,
                          headers={'User-Agent': 'Mozilla/5.0'})
        lat, lon = extract_latlon(r.url)
        if lat is None:
            lat, lon = extract_latlon(r.text[:5000])
        return lat, lon
    except Exception:
        return None, None

for i, s in enumerate(sites_raw, 1):
    lat, lon = resolve_link(s['url'])
    s['lat'], s['lon'] = lat, lon
    print(f"[{i:02d}/32] {s['name']}: {'OK' if lat else 'FAILED'}")
    time.sleep(0.4)

df = pd.DataFrame(sites_raw).dropna(subset=['lat', 'lon']).reset_index(drop=True)
print(f'\nจำนวนจุดที่ resolve พิกัดสำเร็จ: {len(df)}/32')

## 4. หาพิกัดสถานี PCD (24T, 25T) และคำนวณระยะทาง

In [ ]:
PCD_STATIONS = ['24t', '25t']  # 24t = ต.หน้าพระลาน อ.เฉลิมพระเกียรติ, 25t = ต.ปากเพรียว อ.เมือง (สระบุรี)

# พิกัดสำรอง ระดับตำบล (ใช้กรณี Air4Thai API เรียกไม่สำเร็จ) - ความแม่นยำระดับ ~1-2 กม.
# 24t: ศูนย์กลาง ต.หน้าพระลาน/อ.เฉลิมพระเกียรติ (th.wikipedia.org)
# 25t: รพ.สระบุรี ต.ปากเพรียว อ.เมืองสระบุรี (en.wikipedia.org) - อยู่ในตำบลเดียวกับสถานี
FALLBACK_COORDS = {
    '24t': {'lat': 14.6089, 'lon': 100.9047},
    '25t': {'lat': 14.5345, 'lon': 100.9157},
}

station_coords = {}
try:
    resp = requests.get('http://air4thai.pcd.go.th/services/getNewAQI_JSON.php', timeout=20)
    stations_json = resp.json()['stations']
    for st in stations_json:
        sid = st.get('stationID', '').lower()
        if sid in PCD_STATIONS:
            station_coords[sid] = {'lat': float(st['lat']), 'lon': float(st['long'])}
except Exception as e:
    print('⚠️ ดึงพิกัดสถานีอัตโนมัติไม่สำเร็จ:', e)

# เติมพิกัดสำรองสำหรับสถานีที่ยังไม่ได้พิกัดจาก API (ไม่ว่าจะเพราะ API ล้มเหลวทั้งหมด หรือได้มาไม่ครบ)
for sid in PCD_STATIONS:
    if sid not in station_coords:
        station_coords[sid] = FALLBACK_COORDS[sid]
        print(f'ℹ️ ใช้พิกัดสำรองสำหรับ {sid}: {FALLBACK_COORDS[sid]}')

print('พิกัดสถานีที่ใช้:', station_coords)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

for sid, info in station_coords.items():
    df[f'dist_to_{sid}_km'] = haversine_km(df['lat'], df['lon'], info['lat'], info['lon'])

dist_cols = [f'dist_to_{sid}_km' for sid in station_coords]
df['nearest_pcd_station'] = df[dist_cols].idxmin(axis=1).str.replace('dist_to_', '').str.replace('_km', '')
df['dist_to_nearest_pcd_km'] = df[dist_cols].min(axis=1)

df[['name', 'pm25', 'date', 'nearest_pcd_station', 'dist_to_nearest_pcd_km']]

## 5. อัปโหลดไฟล์ PCD (export จาก Air4Thai) และดึงค่าเฉลี่ยรายวัน

ไฟล์นี้เป็นไฟล์เดียวรวมทุกสถานีทั้งเครือข่าย แบบค่าเฉลี่ยรายวัน (ชีต `PM2.5`, คอลัมน์ `Date` + รหัสสถานี) — ดาวน์โหลดได้จาก https://air4thai.pcd.go.th/webV3/#/History

In [ ]:
from google.colab import files
print('อัปโหลดไฟล์ Air4Thai export (เช่น 2023.xlsx):')
pcd_uploaded = files.upload()
PCD_XLSX = list(pcd_uploaded.keys())[0]

# ช่วงวันที่ของแคมเปญ - แก้ตรงนี้ถ้าไม่ตรงกับของคุณ
SDATE = '2023-06-05'
EDATE = '2023-06-29'

raw_pcd = pd.read_excel(PCD_XLSX, sheet_name='PM2.5')
raw_pcd = raw_pcd[pd.to_datetime(raw_pcd['Date'], errors='coerce').notna()].copy()  # กรองแถวหมายเหตุท้ายไฟล์ออก
raw_pcd['Date'] = pd.to_datetime(raw_pcd['Date'])

mask = (raw_pcd['Date'] >= pd.to_datetime(SDATE)) & (raw_pcd['Date'] <= pd.to_datetime(EDATE))
campaign_pcd = raw_pcd.loc[mask].copy()
print(f'พบข้อมูล {len(campaign_pcd)} วัน ในช่วง {SDATE} ถึง {EDATE}')

daily_means = {}
for sid in station_coords:
    col = sid.upper()  # ไฟล์ export ใช้ตัวพิมพ์ใหญ่ เช่น '24T'
    if col not in campaign_pcd.columns:
        print(f'⚠️ ไม่พบคอลัมน์ {col} ในไฟล์')
        continue
    vals = pd.to_numeric(campaign_pcd[col], errors='coerce')  # 'N/A' (ข้อความ) -> NaN
    daily_means[sid] = pd.Series(vals.values, index=campaign_pcd['Date'].dt.date)
    print(f'{sid} ({col}): มีค่า {daily_means[sid].notna().sum()}/{len(campaign_pcd)} วันในช่วงแคมเปญ')

## 6. จับคู่ข้อมูลภาคสนามกับค่าเฉลี่ยรายวันของสถานีที่ใกล้ที่สุด

⚠️ วันที่ในไฟล์ภาคสนามอาจเป็นรูปแบบ `D/M/YY` แบบไทย โดยปี 2 หลักบางแถวเป็น พ.ศ. (เช่น 66 = 2566) และบางแถวเป็น ค.ศ. (เช่น 23 = 2023) แบบไม่สม่ำเสมอ — ฟังก์ชันด้านล่างจะลองทั้งสองแบบแล้วเลือกปีที่สมเหตุสมผล (2015-2035) ให้อัตโนมัติ

In [ ]:
def normalize_date(date_str):
    """แปลง string วันที่เป็น datetime.date - รองรับ D/M/YY แบบไทย (พ.ศ. หรือ ค.ศ. 2 หลัก แบบไม่สม่ำเสมอ)"""
    s = str(date_str).strip()
    m = re.match(r'^(\d{1,2})/(\d{1,2})/(\d{2,4})$', s)
    if m:
        day, month, yy = int(m.group(1)), int(m.group(2)), int(m.group(3))
        candidates = [yy] if yy >= 100 else [yy + 2500 - 543, yy + 2000]
        for year in candidates:
            if 2015 <= year <= 2035:
                try:
                    return pd.Timestamp(year=year, month=month, day=day).date()
                except ValueError:
                    continue
        return None
    d = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if pd.isna(d):
        return None
    d = d.date()
    if d.year > 2400:
        d = d.replace(year=d.year - 543)
    return d

def lookup_pcd_daily(sid, date_str):
    d = normalize_date(date_str)
    if d is None:
        return np.nan
    return daily_means.get(sid, pd.Series(dtype=float)).get(d, np.nan)

df['pcd_daily_mean_pm25'] = df.apply(lambda r: lookup_pcd_daily(r['nearest_pcd_station'], r['date']), axis=1)
df['diff_daily'] = df['pm25'] - df['pcd_daily_mean_pm25']

n_match = df['pcd_daily_mean_pm25'].notna().sum()
print(f'จับคู่สำเร็จ: {n_match} / {len(df)} จุด')
df[['name', 'pm25', 'date', 'nearest_pcd_station', 'dist_to_nearest_pcd_km', 'pcd_daily_mean_pm25', 'diff_daily']]

## 7. สร้าง Table 7 (Pearson / Spearman ทั้ง 3 การเปรียบเทียบ)

In [ ]:
def correlation_row(comparison_label, sub, x_col, y_col):
    sub = sub[[x_col, y_col]].dropna()
    n = len(sub)
    if n < 3:
        return {'Comparison': comparison_label, 'Variable': y_col, 'n': n,
                'Pearson r (p)': 'n/a', 'Spearman rho (p)': 'n/a'}
    r, p = stats.pearsonr(sub[x_col], sub[y_col])
    rho, p_rho = stats.spearmanr(sub[x_col], sub[y_col])
    return {
        'Comparison': comparison_label,
        'Variable': 'Raw PM2.5' if x_col == 'pm25' else 'Log PM2.5',
        'n': n,
        'Pearson r (p)': f'{r:.3f} ({p:.3f})',
        'Spearman rho (p)': f'{rho:.3f} ({p_rho:.3f})',
    }

df_valid = df.dropna(subset=['pm25', 'pcd_daily_mean_pm25']).copy()
df_log = df_valid.copy()
df_log['log_pm25'] = np.log(df_log['pm25'])
df_log['log_pcd'] = np.log(df_log['pcd_daily_mean_pm25'])

df_excl = df_valid[df_valid['pm25'] < 400]  # ไม่รวม Tab Kwang (extreme value)

table7_rows = [
    correlation_row('All sites', df_valid, 'pm25', 'pcd_daily_mean_pm25'),
]
row_log = correlation_row('All sites', df_log, 'log_pm25', 'log_pcd')
row_log['Spearman rho (p)'] = '—'  # รายงานเฉพาะ Pearson สำหรับ log-scale ตามต้นฉบับ
table7_rows.append(row_log)
table7_rows.append(correlation_row('Excl. Tab Kwang', df_excl, 'pm25', 'pcd_daily_mean_pm25'))

table7 = pd.DataFrame(table7_rows)
print(f"ระยะทางเฉลี่ยไปสถานี PCD ที่ใกล้ที่สุด: {df['dist_to_nearest_pcd_km'].mean():.1f} กม. "
      f"(ช่วง {df['dist_to_nearest_pcd_km'].min():.1f}–{df['dist_to_nearest_pcd_km'].max():.1f} กม.)")
print(f"จำนวนจุดที่ใกล้ 24T ที่สุด: {(df['nearest_pcd_station']=='24t').sum()}, "
      f"ใกล้ 25T ที่สุด: {(df['nearest_pcd_station']=='25t').sum()}\n")
table7

## 8. กราฟ scatter ประกอบ (field vs. PCD daily mean)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = df_valid['nearest_pcd_station'].map({'24t': '#e6550d', '25t': '#3182bd'})
ax.scatter(df_valid['pcd_daily_mean_pm25'], df_valid['pm25'], c=colors, edgecolors='black', alpha=0.85)
lims = [0, max(df_valid['pm25'].max(), df_valid['pcd_daily_mean_pm25'].max()) * 1.1]
ax.plot(lims, lims, 'k--', alpha=0.5, label='1:1 reference')
ax.set_xlabel('PCD nearest-station daily-mean PM2.5 (µg/m³)')
ax.set_ylabel('Field-measured PM2.5 (µg/m³)')
ax.set_title('Field vs. nearest PCD reference station (daily mean)')
ax.legend()
plt.tight_layout()
plt.savefig('Figure_Table7_PCD_crossvalidation.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. ดาวน์โหลดผลลัพธ์

In [ ]:
from google.colab import files

table7.to_csv('Table7_pcd_crossvalidation.csv', index=False)
table7.to_excel('Table7_pcd_crossvalidation.xlsx', index=False)
df[['name', 'pm25', 'date', 'lat', 'lon', 'nearest_pcd_station', 'dist_to_nearest_pcd_km',
    'pcd_daily_mean_pm25', 'diff_daily']].to_excel('Table7_site_level_detail.xlsx', index=False)

files.download('Table7_pcd_crossvalidation.csv')
files.download('Table7_pcd_crossvalidation.xlsx')
files.download('Table7_site_level_detail.xlsx')
files.download('Figure_Table7_PCD_crossvalidation.png')

---
✅ **เสร็จสิ้น** — ผลลัพธ์ควรตรงกับ Table 7 ในต้นฉบับ: All sites (n=32) r=0.253 (p=0.162), Log PM2.5 r=0.324 (p=0.071), Excl. Tab Kwang (n=31) r=0.235 (p=0.203)

ถ้าตัวเลขที่ได้ไม่ตรงกับต้นฉบับ ให้ตรวจสอบ:
- ไฟล์ Air4Thai export ที่ใช้เป็นไฟล์เดียวกับที่ใช้สร้างตารางในต้นฉบับหรือไม่
- จำนวนจุดที่ resolve พิกัดสำเร็จใน Section 3 ครบ 32 จุดหรือไม่